# Panel de indicadores CAMELS

Apila los archivos `indicad/{codigo}.txt` de los 133 dumps IEF. Cada archivo trae los ~40 indicadores CAMELS calculados por el BCRA para esa entidad, en formato wide con 5 fechas por columna.

Las 5 fechas se leen de `entidad/{codigo}.txt` del mismo dump (columnas 13-17, en formato yyyymm).

Output: `data/interim/paneles_largos/panel_indicadores.parquet` en formato long, con columnas:
- `codigo_entidad`, `nombre_entidad`
- `yyyymm` (fecha del valor del indicador)
- `codigo_linea`, `descripcion_indicador`
- `valor`
- `valor_grupo_homogeneo`, `valor_top10_privados`, `valor_sistema_financiero` (benchmarks)
- `formato_valor` ('P' porcentaje o 'N' número)
- `dump_yyyymm` (procedencia)

Decisión: cuando una observación `(entidad, yyyymm, codigo_linea)` aparece en múltiples dumps, me quedo con la del dump más reciente.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import RAW, INTERIM, DIMENSIONES, PANELES, REPO

import pandas as pd
import duckdb

IEF_ROOT = RAW / "bcra_ief"
OUT = PANELES / "panel_indicadores.parquet"
OUT.parent.mkdir(parents=True, exist_ok=True)

# Indicad: 5 fechas en cols 5-9; entidad/cod.txt: yyyymm en cols 12-16 (índice 0)
INDICAD_FECHA_COLS = ["v1", "v2", "v3", "v4", "v5"]
ENTIDAD_FECHA_IDX = list(range(12, 17))  # 13-17 en notación 1-based

## Helper: leer fechas del archivo entidad/

Cada `entidad/{cod}.txt` trae las 5 fechas que corresponden a los 5 valores en `indicad/`.

In [ ]:
def leer_fechas_entidad(entidad_file):
    """Devuelve lista de 5 yyyymm (str) leídas del archivo entidad/{cod}.txt"""
    texto = entidad_file.read_text(encoding="ISO-8859-1")
    campos = [c.strip().strip('"') for c in texto.split("\t")]
    if len(campos) <= ENTIDAD_FECHA_IDX[-1]:
        return []
    return [campos[i] for i in ENTIDAD_FECHA_IDX]

## Helper: leer un archivo indicad/{cod}.txt y melt

In [ ]:
def leer_indicad(indicad_file, fechas_5):
    cols = ["codigo_entidad", "nombre_entidad", "fecha_corte", "codigo_linea",
            "descripcion_indicador", "v1", "v2", "v3", "v4", "v5",
            "valor_grupo_homogeneo", "valor_top10_privados", "valor_sistema_financiero",
            "formato_valor"]
    df = pd.read_csv(indicad_file, sep="\t", header=None, names=cols,
                     encoding="ISO-8859-1", dtype=str)
    for c in INDICAD_FECHA_COLS + ["valor_grupo_homogeneo", "valor_top10_privados", "valor_sistema_financiero"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Mapeo de v1..v5 a las fechas
    mapeo = dict(zip(INDICAD_FECHA_COLS, fechas_5))

    long = df.melt(
        id_vars=["codigo_entidad", "nombre_entidad", "codigo_linea", "descripcion_indicador",
                 "valor_grupo_homogeneo", "valor_top10_privados", "valor_sistema_financiero",
                 "formato_valor"],
        value_vars=INDICAD_FECHA_COLS,
        var_name="slot_fecha", value_name="valor"
    )
    long["yyyymm"] = long["slot_fecha"].map(mapeo)
    long = long.drop(columns=["slot_fecha"]).dropna(subset=["yyyymm"])
    return long

## Apilado de los 133 dumps

In [ ]:
dumps = sorted([d for d in IEF_ROOT.iterdir() if d.is_dir() and d.name.isdigit()])
bloques = []
for d in dumps:
    yyyymm_dump = d.name
    indicad_dir = d / "Entfin/Tec_Cont/indicad"
    entidad_dir = d / "Entfin/Tec_Cont/entidad"
    if not indicad_dir.exists() or not entidad_dir.exists():
        continue

    for indicad_file in indicad_dir.glob("*.txt"):
        if indicad_file.name == "formato.txt":
            continue
        cod = indicad_file.stem
        entidad_file = entidad_dir / f"{cod}.txt"
        if not entidad_file.exists():
            continue
        try:
            fechas = leer_fechas_entidad(entidad_file)
            if len(fechas) != 5:
                continue
            long = leer_indicad(indicad_file, fechas)
            long["dump_yyyymm"] = int(yyyymm_dump)
            bloques.append(long)
        except Exception as e:
            pass

raw = pd.concat(bloques, ignore_index=True)
print(f"Filas apiladas: {len(raw):,}")

## Deduplicación

Cada `(entidad, yyyymm, codigo_linea)` puede aparecer en múltiples dumps. Me quedo con la versión del dump más reciente. Antes filtro filas con yyyymm inválido (algunos bancos nuevos o dados de baja tienen menos de 5 fechas reportadas).

In [ ]:
raw = raw[raw["yyyymm"].astype(str).str.match(r"^\d{6}$")].copy()
raw["yyyymm"] = raw["yyyymm"].astype(int)
raw = raw.sort_values(["codigo_entidad", "yyyymm", "codigo_linea", "dump_yyyymm"])
panel = raw.drop_duplicates(subset=["codigo_entidad", "yyyymm", "codigo_linea"], keep="last")
print(f"Filas tras deduplicar: {len(panel):,}")

## Persistencia

In [ ]:
panel.to_parquet(OUT, index=False)
print(f"Escrito: {OUT.relative_to(REPO)}")
print(f"Filas: {len(panel):,}")

## Validaciones

In [ ]:
assert panel[["codigo_entidad", "yyyymm", "codigo_linea"]].duplicated().sum() == 0
print(f"Validaciones OK")
print(f"  entidades: {panel['codigo_entidad'].nunique()}")
print(f"  indicadores distintos: {panel['codigo_linea'].nunique()}")
print(f"  rango fechas: {panel['yyyymm'].min()} - {panel['yyyymm'].max()}")

## Muestra: indicador C1 (apalancamiento) por banco

In [ ]:
duckdb.sql(f"""
    select codigo_entidad, nombre_entidad, yyyymm, valor
    from '{OUT}'
    where descripcion_indicador like 'C1%'
      and codigo_entidad in ('00007', '00011', '00072')
      and yyyymm >= 202401
    order by codigo_entidad, yyyymm
    limit 15
""").df()